# 07 - Athena Database Creation
The goal of this notebook is to demonstrate the use of Amazon Glue and Amazon Athena to build a database that enables TalentIQ to efficiently generate external job market insights from LinkedIn job postings. These insights can be leveraged to enhance and inform the attrition prediction model over time.

In [6]:
import boto3
import sagemaker
import time

session = sagemaker.Session()
region = session.boto_region_name
role_arn = sagemaker.get_execution_role()

glue = boto3.client("glue", region_name=region)
athena = boto3.client("athena", region_name=region)

bucket = "ads508-team-4-private"
glue_db = "talentiq_hr_db"
crawler_name = "talentiq_linkedin_crawler"
athena_output = f"s3://{bucket}/athena-results/"

print("Region:", region)
print("Role ARN:", role_arn)
print("Athena output:", athena_output)

Region: us-east-1
Role ARN: arn:aws:iam::802896854792:role/LabRole
Athena output: s3://ads508-team-4-private/athena-results/


In [8]:
try:
    glue.create_database(
        DatabaseInput={
            "Name": glue_db,
            "Description": "External databse for TalentIQ"
        }
    )
    print("Created database:", glue_db)
except glue.exceptions.AlreadyExistsException:
    print("Database already exists:", glue_db)

Database already exists: talentiq_hr_db


In [9]:
role_name = role_arn.split("/")[-1]

crawler_config = {
    "Name": crawler_name,
    "Role": role_name,
    "DatabaseName": glue_db,
    "Targets": {
        "S3Targets": [
            {"Path": f"s3://{bucket}/raw/external/"}
        ]
    },
    "SchemaChangePolicy": {
        "UpdateBehavior": "UPDATE_IN_DATABASE",
        "DeleteBehavior": "LOG"
    }
}

try:
    glue.create_crawler(**crawler_config)
    print("Created crawler:", crawler_name)
except glue.exceptions.AlreadyExistsException:
    print("Crawler already exists:", crawler_name)

Crawler already exists: talentiq_linkedin_crawler


In [32]:
try:
    glue.start_crawler(Name=crawler_name)
    print("Started crawler:", crawler_name)
except glue.exceptions.CrawlerRunningException:
    print("Crawler already running.")

while True:
    crawler = glue.get_crawler(Name=crawler_name)["Crawler"]
    state = crawler["State"]
    print("Crawler state:", state)
    if state == "READY":
        break
    time.sleep(10)

Started crawler: talentiq_linkedin_crawler
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: READY


In [33]:
tables = glue.get_tables(DatabaseName=glue_db)["TableList"]

print("Tables in Glue database:")
for t in tables:
    print("-", t["Name"])

Tables in Glue database:
- 1
- 2
- 3
- 4
- 5
- 6
- 7
- benefits
- companies
- company_industries
- company_specialties
- job_industries
- job_postings
- job_skills


In [35]:
table_name = "company_specialties"
table_info = glue.get_table(DatabaseName=glue_db, Name=table_name)
print(table_info["Table"]["StorageDescriptor"]["Columns"])

[{'Name': 'company_id', 'Type': 'bigint'}, {'Name': 'speciality', 'Type': 'string'}]


In [14]:
def run_athena_query(query, database=glue_db, output=athena_output):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output}
    )
    query_id = response["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]["Status"]["State"]
        if status in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break
        time.sleep(2)

    print("Query ID:", query_id)
    print("Status:", status)

    if status != "SUCCEEDED":
        reason = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]["Status"].get("StateChangeReason", "")
        raise RuntimeError(f"Athena query failed: {reason}")

    return query_id

In [30]:
run_athena_query("""
SELECT company_id, speciality
FROM company_specialties
LIMIT 10
""")

Query ID: 78cde596-8c72-4aa3-9f01-618eb9e7bb05
Status: SUCCEEDED


'78cde596-8c72-4aa3-9f01-618eb9e7bb05'